# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

The structure of this notebook is as follows: 
1. Data setup: loading features, metadata, splits
2. Training helper: function to train models and save outputs
3. Mean baseline 
4. Multiple linear regression
5. Random forest
6. 1D-CNN

## 1. Data setup

In [1]:
# imports and setting file paths
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyRegressor
import tensorflow as tf

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task2"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models 
from src.data import load_metadata, col_names
from src.features import X as features
from src.models.linreg import make_linear_regression
from src.models.rf import make_random_forest
from src.models.cnn1d import build_cnn

print("Features shape:", features.shape)
display(features.head())

Features shape: (165, 5)


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
GaCo01,1.230322,0.042125,0.054223,0.013902,14.561747
GaCo02,1.103386,0.035667,0.037527,0.001486,15.233429
GaCo03,1.241191,0.049523,0.078598,0.103336,12.754554
GaCo04,1.249965,0.058603,0.064637,0.022956,15.401692
GaCo05,1.051882,0.039561,0.055866,0.013298,12.820625


In [2]:
# load data 
metadata = load_metadata().rename(columns={'ID':'subject'})
features = features.rename_axis('subject').reset_index()

# merge metadata and features into patients 
patients = features.merge(
    metadata[['subject','Group','UPDRSM', 'HoehnYahr']], 
    on='subject', how='inner', validate='many_to_one')
patients = patients.loc[
    patients['Group'].eq('Pt') 
    & patients['UPDRSM'].notna()
    & patients['HoehnYahr'].notna()
    ].reset_index(drop=True)

feature_columns = ['stride_time','stride_cv','swing_cv','swing_asymmetry','peak_force']

X = patients[feature_columns]
y_updrsm = patients['UPDRSM'].astype(float) # for UPDRSM prediction
y_hy = patients['HoehnYahr'].astype(float) # for Hoehn & Yahr prediction 

print("Patients:", patients["subject"].nunique(), "| Feature matrix:", X.shape)
display(
    patients[["subject", "UPDRSM", "HoehnYahr"]].head()
)

Patients: 91 | Feature matrix: (91, 5)


,subject,UPDRSM,HoehnYahr
0,GaPt03,10.0,3.0
1,GaPt04,8.0,2.5
2,GaPt05,5.0,2.5
3,GaPt06,13.0,2.0
4,GaPt07,22.0,3.0


In [3]:
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    
    # safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects) and val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(patients["subject"].isin(train_ids).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(val_ids).to_numpy())
    folds.append((train_idx, val_idx))

print('CV folds:',len(folds),'held-out test subjects:',len(test_subjects))
print("Using", len(folds), "saved validation folds; held-out test subjects untouched.")

CV folds: 5 held-out test subjects: 19
Using 5 saved validation folds; held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output. 

In [4]:
def train_and_save(model_name, make_model, y_target, target_name):
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        model = make_model()  
        model.fit(X.iloc[train_idx], 
                  y_target.iloc[train_idx])  # train only on training subjects
        predictions = np.asarray(model.predict(X.iloc[val_idx])).reshape(-1)

        # Save actual values and predictions for the results teammate.
        output = pd.DataFrame({
            "subject": patients.iloc[val_idx]["subject"].to_numpy(),
            "fold": fold_number,
            f"actual_{target_name}": y_target.iloc[val_idx].to_numpy(),
            f"predicted_{target_name}": predictions
        })
        
        if target_name == "UPDRSM":
            filename = model_name
        else:
            filename = f"{model_name}_{target_name}"
            
        output.to_csv(
            OUT / "predictions" /
            f"{filename}_val_fold{fold_number}.csv",
            index=False
        )
        # save fitted preprocessing steps and model together
        joblib.dump(
            model,
            OUT / "weights" /
            f"{filename}_val_fold{fold_number}.joblib"
        )

        print(
            f"{model_name} ({target_name}): "
            f"fold {fold_number} saved"
        )
    print(f"{model_name}: model train complete")

## 3. Mean baseline 
This is used as a reference point. We will compare our model performance with this baseline. 

In [5]:
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"),
               y_updrsm, "UPDRSM")

mean_baseline (UPDRSM): fold 1 saved
mean_baseline (UPDRSM): fold 2 saved
mean_baseline (UPDRSM): fold 3 saved
mean_baseline (UPDRSM): fold 4 saved
mean_baseline (UPDRSM): fold 5 saved
mean_baseline: model train complete


## 4. Multiple Linear Regression

In [6]:
train_and_save("linear_regression", make_linear_regression,
               y_updrsm, "UPDRSM")

linear_regression (UPDRSM): fold 1 saved
linear_regression (UPDRSM): fold 2 saved
linear_regression (UPDRSM): fold 3 saved
linear_regression (UPDRSM): fold 4 saved
linear_regression (UPDRSM): fold 5 saved
linear_regression: model train complete


## 5. Random Forest

In [7]:
train_and_save("random_forest", make_random_forest, 
               y_updrsm, "UPDRSM")

random_forest (UPDRSM): fold 1 saved
random_forest (UPDRSM): fold 2 saved
random_forest (UPDRSM): fold 3 saved
random_forest (UPDRSM): fold 4 saved
random_forest (UPDRSM): fold 5 saved
random_forest: model train complete


In [8]:
# hyperparameter tuning 

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error

# combinations to test
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, None],
    "model__min_samples_leaf": [2, 5]
}

tuning_results = [] # store results

for params in ParameterGrid(param_grid):
    fold_rmse = []
    
    for train_idx, val_idx in folds:
        # create new model and apply parameter combination 
        model = make_random_forest() 
        model.set_params(**params) 
        model.fit(X.iloc[train_idx], y_updrsm.iloc[train_idx])
        predictions = model.predict(X.iloc[val_idx])

        # calculate validation RMSE
        rmse = np.sqrt(mean_squared_error(y_updrsm.iloc[val_idx],predictions))

        fold_rmse.append(rmse)

    # calculate average validation RMSE across five folds.
    tuning_results.append({
        **params,
        "mean_RMSE": np.mean(fold_rmse),
        "std_RMSE": np.std(fold_rmse)
    })

rf_tuning = pd.DataFrame(tuning_results)
rf_tuning = rf_tuning.sort_values("mean_RMSE")
display(rf_tuning)

print("Best RF hyperparameters:")
display(rf_tuning.head(1))


,model__max_depth,model__min_samples_leaf,model__n_estimators,mean_RMSE,std_RMSE
7,5.0,5,200,8.123274,1.316859
11,NaN,5,200,8.123657,1.315656
3,3.0,5,200,8.145958,1.286554
10,NaN,5,100,8.209255,1.343422
6,5.0,5,100,8.209772,1.345528
1,3.0,2,200,8.212829,1.332050
2,3.0,5,100,8.225235,1.281660
0,3.0,2,100,8.276718,1.321642
5,5.0,2,200,8.322056,1.293030
9,NaN,2,200,8.326657,1.313910


Best RF hyperparameters:


,model__max_depth,model__min_samples_leaf,model__n_estimators,mean_RMSE,std_RMSE
7,5.0,5,200,8.123274,1.316859


In [ ]:
# with tuned hyperparameters
def make_tuned_rf(): # use same hyperparameter tuning for clean comparison between UPDRSM and HY
    model = make_random_forest(
        task="regression",
        n_estimators=200, 
        min_samples_leaf=5,  
        random_state=42
    )
    model.set_params(model__max_depth=5) 
    return model

train_and_save("random_forest_tuned", make_tuned_rf, y_updrsm, "UPDRSM")

random_forest_tuned (UPDRSM): fold 1 saved
random_forest_tuned (UPDRSM): fold 2 saved
random_forest_tuned (UPDRSM): fold 3 saved
random_forest_tuned (UPDRSM): fold 4 saved
random_forest_tuned (UPDRSM): fold 5 saved
random_forest_tuned: model train complete


## 6. 1D-CNN Regression
The CNN model uses 16 force-sensor channels over 1,000 time steps per subject. This is a 10-second starting window. 

In [10]:
PROCESSED = ROOT/'data'/'processed'
RAW = ROOT/'data'/'raw'
TIME_STEPS = 1000
SENSOR_COLS = [f'L{i}' for i in range(1,9)] + [f'R{i}' for i in range(1,9)]

def load_signal(subject):
    csv_path = PROCESSED/f'{subject}_01.csv'
    txt_path = RAW/f'{subject}_01.txt'
    if csv_path.exists():
        walk = pd.read_csv(csv_path)
    else:
        walk = pd.read_csv(txt_path,sep=r'\s+',header=None,names=col_names)
    signal = walk[SENSOR_COLS].iloc[:TIME_STEPS].to_numpy(dtype=np.float32)
    if signal.shape != (TIME_STEPS,16) or not np.isfinite(signal).all():
        raise ValueError(f'Invalid CNN signal for {subject}: {signal.shape}')
    return signal

signals = np.stack([load_signal(subject) for subject in patients['subject']])
print('CNN input shape (subjects, time steps, channels):',signals.shape)

CNN input shape (subjects, time steps, channels): (91, 1000, 16)


In [ ]:
# train CNN model per fold, saving predictions and weights for each
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
EPOCHS = 20
BATCH_SIZE = 16
for fold_number, (train_idx, val_idx) in enumerate(folds,1):
    # normalise signals 
    mean = signals[train_idx].mean(axis=(0,1),keepdims=True)
    std = np.maximum(signals[train_idx].std(axis=(0,1),keepdims=True),1e-6)
    x_train = (signals[train_idx]-mean)/std
    x_val = (signals[val_idx]-mean)/std
    
    # create and train model for each fold 
    model = build_cnn(input_shape=(TIME_STEPS,16),
                      task='regression', learning_rate = 0.001)
    model.fit(x_train,y_updrsm.iloc[train_idx].to_numpy(),
              epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=1)
    

    predicted = model.predict(x_val,verbose=0).reshape(-1)
    pd.DataFrame({'subject':patients.iloc[val_idx]['subject'].to_numpy(),
                  'fold':fold_number,'actual_UPDRSM':y_updrsm.iloc[val_idx].to_numpy(),
                  'predicted_UPDRSM':predicted}).to_csv(
        OUT/'predictions'/f'cnn_val_fold{fold_number}.csv',index=False)
    model.save(OUT/'weights'/f'cnn_val_fold{fold_number}.keras')
    
    # save predictions and model weights
    np.savez(OUT/'weights'/f'cnn_val_fold{fold_number}_normalisation.npz',
             mean=mean,std=std,time_steps=TIME_STEPS,sensor_columns=np.array(SENSOR_COLS))
    print(f'Saved CNN fold {fold_number} ({len(val_idx)} predictions)')
print("CNN-1D model training complete.")

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 410.5540 - mae: 18.5496
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 383.5503 - mae: 17.7956 
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 352.5683 - mae: 16.8850 
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 314.0274 - mae: 15.6758 
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 266.4514 - mae: 14.0876
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 211.2264 - mae: 12.1071 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 153.8764 - mae: 9.7495 
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 106.0469 - mae: 7.6712
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 82.7157 - mae: 7.1103 
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 84.1473 - mae: 7.5378 
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 86.7039 - mae: 7.7381 
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 80.5525 - mae: 7.3755 
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━━

In [19]:
# hyperparameter tuning 
learning_rates = [0.001, 0.0001]
epoch_options = [10, 20, 30]

cnn_tuning_results = []

for learning_rate in learning_rates:
    for epochs in epoch_options:

        fold_rmse = []
        
        for fold_number, (train_idx, val_idx) in enumerate(folds):
            tf.keras.utils.set_random_seed(42)

            # nomralisation
            mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
            std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
            x_train = (signals[train_idx] - mean) / std
            x_val = (signals[val_idx] - mean) / std

            # build and train new cnn
            model = build_cnn(
                input_shape=(TIME_STEPS, 16),
                task="regression",
                learning_rate=learning_rate)
            
            model.fit(x_train, y_updrsm.iloc[train_idx].to_numpy(),
                epochs=epochs, batch_size=16, verbose=0)

            # generate validation predictions
            predictions = model.predict(x_val, verbose=0).reshape(-1)

            # calculate validation RMSE
            rmse = np.sqrt(mean_squared_error(y_updrsm.iloc[val_idx], predictions))
            fold_rmse.append(rmse)

        cnn_tuning_results.append({
            "learning_rate": learning_rate,
            "epochs": epochs,
            "mean_RMSE": np.mean(fold_rmse),
            "std_RMSE": np.std(fold_rmse)
        })

# compare configurations
cnn_tuning = pd.DataFrame(cnn_tuning_results)
cnn_tuning = cnn_tuning.sort_values("mean_RMSE")

display(cnn_tuning)

print("Best CNN configuration:")
display(cnn_tuning.head(1))

,learning_rate,epochs,mean_RMSE,std_RMSE
2,0.0010,30,8.671473,1.516047
1,0.0010,20,8.676058,1.468264
0,0.0010,10,9.400274,1.941435
5,0.0001,30,17.005467,1.840138
4,0.0001,20,18.688148,1.830413
3,0.0001,10,19.737863,1.814449


Best CNN configuration:


,learning_rate,epochs,mean_RMSE,std_RMSE
2,0.001,30,8.671473,1.516047


In [20]:
# CNN with tuned hyperparameters
TUNED_EPOCHS = 30
TUNED_LEARNING_RATE = 0.001
BATCH_SIZE = 16

for fold_number, (train_idx, val_idx) in enumerate(folds, 1):
    tf.keras.utils.set_random_seed(42)
    # normalisation
    mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
    std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
    x_train = (signals[train_idx] - mean) / std
    x_val = (signals[val_idx] - mean) / std

    # build cnn
    model = build_cnn(input_shape=(TIME_STEPS, 16), task="regression",
                      learning_rate=TUNED_LEARNING_RATE
    )

    # train and predict
    model.fit(x_train, y_updrsm.iloc[train_idx].to_numpy(),
              epochs=TUNED_EPOCHS, batch_size=BATCH_SIZE, verbose=1)

    predictions = model.predict(x_val, verbose=0).reshape(-1)

    # save output
    pd.DataFrame({
        "subject": patients.iloc[val_idx]["subject"].to_numpy(),
        "fold": fold_number,
        "actual_UPDRSM": y_updrsm.iloc[val_idx].to_numpy(),
        "predicted_UPDRSM": predictions
    }).to_csv(
        OUT / "predictions" /
        f"cnn_tuned_val_fold{fold_number}.csv",
        index=False
    )
    model.save(
        OUT / "weights" /
        f"cnn_tuned_val_fold{fold_number}.keras"
    )
    np.savez(
        OUT / "weights" /
        f"cnn_tuned_val_fold{fold_number}_normalisation.npz",
        mean=mean,
        std=std,
        time_steps=TIME_STEPS,
        sensor_columns=np.array(SENSOR_COLS)
    )

    print(f"Tuned CNN fold {fold_number} saved.")


Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 410.5540 - mae: 18.5496  
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 383.5503 - mae: 17.7956 
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 352.5683 - mae: 16.8850 
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 314.0274 - mae: 15.6758 
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 266.4514 - mae: 14.0876 
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 211.2264 - mae: 12.1071 
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 153.8764 - mae: 9.7495 
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 106.0469 - mae: 7.6712
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 82.7157 - mae: 7.1103 
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 84.1473 - mae: 7.5378 
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 86.7039 - mae: 7.7381 
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 80.5525 - mae: 7.3755 
Epoch 13/30
4/4 ━━━━━━━━━━━━━━━━━━

## 7. Additional Severity Metric - Hoehn & Yahr

Note: Hoehn & Yahr is an ordinal classification problem, not a regression problem. The differences between stages is not necessarily equivalent to one UPDRSM point.

In [21]:
# mean baseline 
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"), 
               y_hy, "HoehnYahr")

mean_baseline (HoehnYahr): fold 1 saved
mean_baseline (HoehnYahr): fold 2 saved
mean_baseline (HoehnYahr): fold 3 saved
mean_baseline (HoehnYahr): fold 4 saved
mean_baseline (HoehnYahr): fold 5 saved
mean_baseline: model train complete


In [22]:
# MLR 
train_and_save("linear_regression", make_linear_regression,
                y_hy, "HoehnYahr")

linear_regression (HoehnYahr): fold 1 saved
linear_regression (HoehnYahr): fold 2 saved
linear_regression (HoehnYahr): fold 3 saved
linear_regression (HoehnYahr): fold 4 saved
linear_regression (HoehnYahr): fold 5 saved
linear_regression: model train complete


In [ ]:
train_and_save("random_forest", make_tuned_rf,
                y_hy, "HoehnYahr")

random_forest (HoehnYahr): fold 1 saved
random_forest (HoehnYahr): fold 2 saved
random_forest (HoehnYahr): fold 3 saved
random_forest (HoehnYahr): fold 4 saved
random_forest (HoehnYahr): fold 5 saved
random_forest: model train complete


In [24]:
# CNN 
HY_EPOCHS = 30
HY_LEARNING_RATE = 0.001

for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

    tf.keras.utils.set_random_seed(42)

    # normalisation
    mean = signals[train_idx].mean(axis=(0, 1), keepdims=True)
    std = np.maximum(signals[train_idx].std(axis=(0, 1), keepdims=True), 1e-6)
    x_train = (signals[train_idx] - mean) / std
    x_val = (signals[val_idx] - mean) / std

    # build new cnn 
    model = build_cnn(
        input_shape=(TIME_STEPS, 16),
        task="regression",
        learning_rate=HY_LEARNING_RATE
    )

    # train and predict
    model.fit(x_train, y_hy.iloc[train_idx].to_numpy(),
        epochs=HY_EPOCHS, batch_size=16, verbose=1)

    predictions = model.predict(x_val, verbose=0).reshape(-1)

    # save
    pd.DataFrame({
        "subject": patients.iloc[val_idx]["subject"].to_numpy(),
        "fold": fold_number,
        "actual_HoehnYahr": y_hy.iloc[val_idx].to_numpy(),
        "predicted_HoehnYahr": predictions
    }).to_csv(
        OUT / "predictions" /
        f"cnn_HoehnYahr_val_fold{fold_number}.csv",
        index=False
    )

    model.save(
        OUT / "weights" /
        f"cnn_HoehnYahr_val_fold{fold_number}.keras"
    )

    np.savez(
        OUT / "weights" /
        f"cnn_HoehnYahr_val_fold{fold_number}_normalisation.npz",
        mean=mean,
        std=std,
        time_steps=TIME_STEPS,
        sensor_columns=np.array(SENSOR_COLS)
    )

    print(f"HoehnYahr CNN fold {fold_number} saved.")


Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 3.8779 - mae: 1.9192 
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.6274 - mae: 1.1672 
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.4681 - mae: 0.5127 
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5127 - mae: 0.6064 
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.4791 - mae: 0.6011 
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2612 - mae: 0.4005 
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2360 - mae: 0.3626 
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.2883 - mae: 0.3970 
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2870 - mae: 0.3926 
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2469 - mae: 0.3670 
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2229 - mae: 0.3638 
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2153 - mae: 0.3618 
Epoch 13/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2